# ChainSight — Data Cleaning & Preparation

## Objective

This notebook transforms the raw DataCo Smart Supply Chain dataset into a clean analytical dataset for exploratory analysis, SQL analytics, Power BI, and predictive modeling.

The raw dataset is preserved unchanged. All transformations are performed on a working copy and the resulting analytical dataset is saved separately.

## Cleaning Principles

- Preserve the raw source data
- Remove unusable and sensitive fields
- Handle missing values based on business context
- Convert dates to appropriate datetime types
- Consolidate redundant identifiers
- Create reproducible analytical features
- Avoid target leakage in predictive features
- Validate the processed dataset before saving

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/raw/DataCoSupplyChainDataset.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

print(f"Raw dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")

Raw dataset loaded: 180,519 rows × 53 columns


In [2]:
df_clean = df.copy()

print(f"Working dataset created: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Working dataset created: 180,519 rows × 53 columns


In [3]:
drop_columns = [
    "Product Description",
    "Customer Password",
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Product Image"
]

df_clean = df_clean.drop(columns=drop_columns)

print(f"Columns after removing unusable/sensitive fields: {df_clean.shape[1]}")

Columns after removing unusable/sensitive fields: 46


In [4]:
redundant_columns = [
    "Order Customer Id",
    "Order Item Cardprod Id"
]

df_clean = df_clean.drop(columns=redundant_columns)

print(f"Columns after removing redundant identifiers: {df_clean.shape[1]}")

Columns after removing redundant identifiers: 44


In [5]:
df_clean = df_clean.drop(columns=["Order Zipcode"])

print(f"Columns after removing incomplete zipcode field: {df_clean.shape[1]}")

Columns after removing incomplete zipcode field: 43


In [6]:
print(
    "Missing Customer Zipcode values:",
    df_clean["Customer Zipcode"].isna().sum()
)

Missing Customer Zipcode values: 3


In [7]:
df_clean = df_clean.drop(columns=["Product Status"])

print(f"Columns after removing constant field: {df_clean.shape[1]}")

Columns after removing constant field: 42


In [8]:
print(df_clean.shape)
print(df_clean.columns.tolist())

(180519, 42)
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'order date (DateOrders)', 'Order Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'shipping date (DateOrders)', 'Shipping Mode']


## Date Transformation

The order and shipping date fields are stored as strings in the raw dataset. They are converted to datetime format to support temporal analysis, trend analysis, and demand forecasting.

The raw date values are preserved in the source dataset.

In [9]:
date_columns = [
    "order date (DateOrders)",
    "shipping date (DateOrders)"
]

for col in date_columns:
    df_clean[col] = pd.to_datetime(
        df_clean[col],
        errors="coerce"
    )

print(df_clean[date_columns].dtypes)

order date (DateOrders)       datetime64[us]
shipping date (DateOrders)    datetime64[us]
dtype: object


In [10]:
for col in date_columns:
    print(
        f"{col} → missing dates:",
        df_clean[col].isna().sum()
    )

order date (DateOrders) → missing dates: 0
shipping date (DateOrders) → missing dates: 0


In [11]:
import re

def to_snake_case(column):
    column = column.strip().lower()
    column = re.sub(r"[^a-z0-9]+", "_", column)
    column = column.strip("_")
    return column

df_clean.columns = [
    to_snake_case(col)
    for col in df_clean.columns
]

print(df_clean.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date_dateorders', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'shipping_date_dateorders', 'shipping_mode']


In [12]:
print(df_clean.shape)
print(df_clean.dtypes)
print(df_clean.columns.tolist())

(180519, 42)
type                                      str
days_for_shipping_real                  int64
days_for_shipment_scheduled             int64
benefit_per_order                     float64
sales_per_customer                    float64
delivery_status                           str
late_delivery_risk                      int64
category_id                             int64
category_name                             str
customer_city                             str
customer_country                          str
customer_id                             int64
customer_segment                          str
customer_state                            str
customer_zipcode                      float64
department_id                           int64
department_name                           str
latitude                              float64
longitude                             float64
market                                    str
order_city                                str
order_country        

In [13]:
missing_summary = (
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[missing_summary > 0]

print(missing_summary)

customer_zipcode    3
dtype: int64


## Missing Value Handling

After removing fields with extensive missingness and unusable information, only three missing values remain in `customer_zipcode`.

These values are retained as missing because postal code is not a core analytical variable for the project, and imputing geographic identifiers could introduce incorrect information.

No other missing values require treatment.

In [14]:
print("Remaining missing values:")
print(df_clean.isna().sum().sum())

Remaining missing values:
3


## Date Feature Engineering

Additional calendar features are derived from the order date to support temporal analysis, trend analysis, seasonality analysis, and demand forecasting.

The original order and shipping datetime fields are retained.

In [15]:
df_clean["order_year"] = df_clean["order_date_dateorders"].dt.year
df_clean["order_month"] = df_clean["order_date_dateorders"].dt.month
df_clean["order_quarter"] = df_clean["order_date_dateorders"].dt.quarter
df_clean["order_day_of_week"] = df_clean["order_date_dateorders"].dt.dayofweek
df_clean["order_day"] = df_clean["order_date_dateorders"].dt.day

print(
    df_clean[
        [
            "order_date_dateorders",
            "order_year",
            "order_month",
            "order_quarter",
            "order_day_of_week",
            "order_day"
        ]
    ].head()
)

  order_date_dateorders  order_year  order_month  order_quarter  \
0   2018-01-31 22:56:00        2018            1              1   
1   2018-01-13 12:27:00        2018            1              1   
2   2018-01-13 12:06:00        2018            1              1   
3   2018-01-13 11:45:00        2018            1              1   
4   2018-01-13 11:24:00        2018            1              1   

   order_day_of_week  order_day  
0                  2         31  
1                  5         13  
2                  5         13  
3                  5         13  
4                  5         13  


## Shipping Duration Feature

The elapsed time between the order date and shipping date is derived as an analytical feature.

This feature describes the observed time between order placement and shipment and can support operational and delivery-performance analysis.

It is retained for descriptive analytics but should not be used as a pre-delivery predictive feature because it depends on the observed shipping date.

In [16]:
df_clean["order_to_shipping_days"] = (
    df_clean["shipping_date_dateorders"]
    - df_clean["order_date_dateorders"]
).dt.total_seconds() / (24 * 60 * 60)

print(df_clean["order_to_shipping_days"].describe())

count    180519.000000
mean          3.498826
std           1.617394
min           0.500000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: order_to_shipping_days, dtype: float64


In [17]:
print("Missing values after feature engineering:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print("\nCurrent shape:")
print(df_clean.shape)

Missing values after feature engineering:
customer_zipcode    3
dtype: int64

Current shape:
(180519, 48)


## Redundancy Check

Several financial fields describe related aspects of order-item economics. Their relationships are examined before deciding whether any fields are redundant.

No fields are removed solely based on similar names or business meaning.

In [18]:
financial_columns = [
    "sales",
    "sales_per_customer",
    "order_item_total",
    "benefit_per_order",
    "order_profit_per_order",
    "order_item_product_price",
    "product_price"
]

financial_summary = df_clean[financial_columns].describe().T

print(financial_summary)

                             count        mean         std         min  \
sales                     180519.0  203.772096  132.273077     9.99000   
sales_per_customer        180519.0  183.107609  120.043670     7.49000   
order_item_total          180519.0  183.107609  120.043670     7.49000   
benefit_per_order         180519.0   21.974989  104.433526 -4274.97998   
order_profit_per_order    180519.0   21.974989  104.433526 -4274.97998   
order_item_product_price  180519.0  141.232550  139.732492     9.99000   
product_price             180519.0  141.232550  139.732492     9.99000   

                                 25%         50%         75%          max  
sales                     119.980003  199.919998  299.950012  1999.989990  
sales_per_customer        104.379997  163.990005  247.399994  1939.989990  
order_item_total          104.379997  163.990005  247.399994  1939.989990  
benefit_per_order           7.000000   31.520000   64.800003   911.799988  
order_profit_per_order     

In [19]:
print("Unique values:")
for col in financial_columns:
    print(f"{col}: {df_clean[col].nunique():,}")

Unique values:
sales: 193
sales_per_customer: 2,927
order_item_total: 2,927
benefit_per_order: 21,998
order_profit_per_order: 21,998
order_item_product_price: 75
product_price: 75


In [20]:
pairs = [
    ("sales_per_customer", "order_item_total"),
    ("benefit_per_order", "order_profit_per_order"),
    ("order_item_product_price", "product_price")
]

for col1, col2 in pairs:
    equal = df_clean[col1].equals(df_clean[col2])
    print(f"{col1} == {col2}: {equal}")

sales_per_customer == order_item_total: True
benefit_per_order == order_profit_per_order: True
order_item_product_price == product_price: True


## Removing Redundant Fields

The redundancy analysis identified three pairs of fields that are exactly identical row-by-row.

To reduce duplication while preserving the analytical information, one field from each identical pair is retained.

The retained fields use the more direct order-item or product terminology.

In [21]:
redundant_financial_columns = [
    "sales_per_customer",
    "benefit_per_order",
    "order_item_product_price"
]

df_clean = df_clean.drop(columns=redundant_financial_columns)

print(f"Shape after removing redundant fields: {df_clean.shape}")

Shape after removing redundant fields: (180519, 45)


In [22]:
print(df_clean.columns.tolist())

['type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_id', 'customer_segment', 'customer_state', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date_dateorders', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_category_id', 'product_name', 'product_price', 'shipping_date_dateorders', 'shipping_mode', 'order_year', 'order_month', 'order_quarter', 'order_day_of_week', 'order_day', 'order_to_shipping_days']


## Numeric Value Validation

Potentially unusual financial values are inspected before any outlier treatment.

Values are not removed automatically because extreme values may represent legitimate business transactions.

In [23]:
numeric_checks = {
    "order_item_discount": df_clean["order_item_discount"].describe(),
    "order_item_discount_rate": df_clean["order_item_discount_rate"].describe(),
    "order_item_profit_ratio": df_clean["order_item_profit_ratio"].describe(),
    "order_profit_per_order": df_clean["order_profit_per_order"].describe(),
    "order_item_quantity": df_clean["order_item_quantity"].describe()
}

for name, summary in numeric_checks.items():
    print(f"\n{name}")
    print(summary)


order_item_discount
count    180519.000000
mean         20.664741
std          21.800901
min           0.000000
25%           5.400000
50%          14.000000
75%          29.990000
max         500.000000
Name: order_item_discount, dtype: float64

order_item_discount_rate
count    180519.000000
mean          0.101668
std           0.070415
min           0.000000
25%           0.040000
50%           0.100000
75%           0.160000
max           0.250000
Name: order_item_discount_rate, dtype: float64

order_item_profit_ratio
count    180519.000000
mean          0.120647
std           0.466796
min          -2.750000
25%           0.080000
50%           0.270000
75%           0.360000
max           0.500000
Name: order_item_profit_ratio, dtype: float64

order_profit_per_order
count    180519.000000
mean         21.974989
std         104.433526
min       -4274.979980
25%           7.000000
50%          31.520000
75%          64.800003
max         911.799988
Name: order_profit_per_order, dty

In [24]:
high_discount = df_clean[
    df_clean["order_item_discount"] >= 500
]

print(high_discount[
    [
        "order_item_id",
        "product_name",
        "product_price",
        "sales",
        "order_item_discount",
        "order_item_discount_rate",
        "order_item_quantity"
    ]
])

       order_item_id         product_name  product_price       sales  \
12606         171765  SOLE E35 Elliptical     1999.98999  1999.98999   

       order_item_discount  order_item_discount_rate  order_item_quantity  
12606                500.0                      0.25                    1  


In [25]:
print(
    f"Rows with discount >= 500: {len(high_discount):,}"
)

Rows with discount >= 500: 1


## Outlier Validation

The maximum `order_item_discount` value of 500 was investigated.

The value corresponds to a product priced at approximately 1,999.99 with a 25% discount rate. The transaction is therefore consistent with the observed discount structure and is retained.

No arbitrary outlier removal or clipping is applied to the financial variables.

In [26]:
negative_profit = df_clean[
    df_clean["order_profit_per_order"] < 0
]

print(f"Negative-profit rows: {len(negative_profit):,}")
print(
    f"Percentage of rows: "
    f"{len(negative_profit) / len(df_clean) * 100:.2f}%"
)

print(
    negative_profit[
        [
            "order_item_id",
            "product_name",
            "sales",
            "order_item_total",
            "order_item_profit_ratio",
            "order_profit_per_order"
        ]
    ].head(10)
)

Negative-profit rows: 33,784
Percentage of rows: 18.71%
    order_item_id                                   product_name       sales  \
1          179254                                   Smart watch   327.750000   
2          179253                                   Smart watch   327.750000   
15         179240                                   Smart watch   327.750000   
16         179239                                   Smart watch   327.750000   
28         179227                                   Smart watch   327.750000   
33         179222                                   Smart watch   327.750000   
34         179221                                   Smart watch   327.750000   
48          71956               Perfect Fitness Perfect Rip Deck  119.980003   
49         113598  Under Armour Girls' Toddler Spine Surge Runni   79.980003   
53         125135           Nike Men's Dri-FIT Victory Golf Polo  100.000000   

    order_item_total  order_item_profit_ratio  order_profit_per

## Negative Profit Validation

The dataset contains 33,784 order items with negative profit, representing approximately 18.71% of all order items.

These observations are retained because negative profit can represent legitimate loss-making transactions and is directly relevant to the financial risk analysis component of ChainSight.

No outlier removal or clipping is applied to negative-profit observations.

In [27]:
ratio_check = (
    df_clean["order_profit_per_order"]
    / df_clean["order_item_total"]
)

comparison = pd.DataFrame({
    "stored_ratio": df_clean["order_item_profit_ratio"],
    "calculated_ratio": ratio_check
})

comparison["difference"] = (
    comparison["stored_ratio"]
    - comparison["calculated_ratio"]
)

print(comparison["difference"].describe())

count    1.805190e+05
mean     1.972752e-04
std      2.106996e-03
min     -5.505315e-03
25%     -2.856478e-05
50%     -1.393939e-09
75%      2.885990e-05
max      5.461260e-03
Name: difference, dtype: float64


## Profit Ratio Validation

The stored `order_item_profit_ratio` was compared with a ratio calculated from `order_profit_per_order` and `order_item_total`.

The differences are small, with a median difference close to zero. The original ratio is therefore retained rather than overwritten, preserving the source dataset's financial calculations.

In [28]:
print("Final shape:", df_clean.shape)

print("\nMissing values:")
print(df_clean.isna().sum()[df_clean.isna().sum() > 0])

print("\nDuplicate rows:", df_clean.duplicated().sum())

print("\nData types:")
print(df_clean.dtypes.value_counts())

Final shape: (180519, 45)

Missing values:
customer_zipcode    3
dtype: int64

Duplicate rows: 0

Data types:
str               16
int64             11
float64           11
int32              5
datetime64[us]     2
Name: count, dtype: int64


In [29]:
print("Order date range:")
print(df_clean["order_date_dateorders"].min())
print(df_clean["order_date_dateorders"].max())

print("\nShipping date range:")
print(df_clean["shipping_date_dateorders"].min())
print(df_clean["shipping_date_dateorders"].max())

print("\nTarget distribution:")
print(df_clean["late_delivery_risk"].value_counts())
print("\nTarget percentages:")
print(
    df_clean["late_delivery_risk"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Order date range:
2015-01-01 00:00:00
2018-01-31 23:38:00

Shipping date range:
2015-01-03 00:00:00
2018-02-06 22:14:00

Target distribution:
late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Target percentages:
late_delivery_risk
1    54.83
0    45.17
Name: proportion, dtype: float64


In [30]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [31]:
OUTPUT_PATH = "../data/processed/chainsight_analytical.csv"

df_clean.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved cleaned dataset to: {OUTPUT_PATH}")

Saved cleaned dataset to: ../data/processed/chainsight_analytical.csv


In [32]:
print(
    f"Saved dataset shape: "
    f"{pd.read_csv(OUTPUT_PATH, encoding='latin1').shape}"
)

Saved dataset shape: (180519, 45)
